In [25]:
!pip -q install pypdf sentence-transformers gradio

In [26]:
import gradio as gr

project_info = """
# 🧠 Hallucination-Aware RAG Assistant

### 📌 Project Overview
An AI-powered document question-answering system that retrieves relevant
information from uploaded PDFs and provides grounded answers with source
citations and confidence checking.

### 🔄 Workflow
PDF Upload
→ Text Extraction
→ Text Chunking
→ Semantic + Keyword Retrieval
→ Relevant Context
→ Grounded Answer
→ Hallucination Check
→ Page Citation

### 🚀 Features
- Multiple PDF upload
- PDF-based question answering
- Semantic search
- Keyword search
- Hybrid retrieval
- Chat memory
- Hallucination/confidence detection
- Page-level source citations
- Multiple answer modes
- Document summary
- Key points extraction
- Suggested questions
- PDF comparison
- Retrieval analytics
- Chat export

### 🛠️ Technologies
- Python
- Google Colab
- Gradio
- PyPDF
- Sentence Transformers
- NumPy
- NLP
- Retrieval-Augmented Generation (RAG)

### 🎯 Main Objective
Reduce unsupported answers by grounding responses in information retrieved
directly from the user's uploaded documents.
"""

with gr.Blocks(title="Project Information") as info_app:
    gr.Markdown(project_info)

info_app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fcda1323fd36f75eba.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import os, re, numpy as np
import gradio as gr
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

# =========================
# MODEL + DATA
# =========================

model = SentenceTransformer("all-MiniLM-L6-v2")

chunks = []
embeddings = None
documents = {}
memory = []

STOP = {
    "the","is","are","was","were","a","an","and","or","of","to",
    "in","on","for","with","what","which","who","when","where",
    "why","how","does","do","did","this","that","these","those",
    "about","from","by"
}

# =========================
# TEXT PROCESSING
# =========================

def get_words(text):
    return set(
        re.findall(r"\b[a-zA-Z0-9]+\b", text.lower())
    ) - STOP


def split_chunks(text, size=120, overlap=25):
    words = text.split()
    result = []

    step = size - overlap

    for i in range(0, len(words), step):
        part = " ".join(words[i:i+size])
        if part.strip():
            result.append(part)

    return result


# =========================
# PDF UPLOAD + PROCESS
# =========================

def upload_pdfs(files):

    global chunks, embeddings, documents, memory

    chunks = []
    embeddings = None
    documents = {}
    memory = []

    if not files:
        return "❌ Please upload a PDF.", "", "", ""

    for file in files:

        path = file if isinstance(file, str) else file.name
        name = os.path.basename(path)

        try:
            reader = PdfReader(path)
        except Exception as e:
            continue

        documents[name] = ""

        for page_no, page in enumerate(reader.pages, 1):

            text = page.extract_text() or ""
            text = re.sub(r"\s+", " ", text).strip()

            if not text:
                continue

            documents[name] += " " + text

            for part in split_chunks(text):

                chunks.append({
                    "file": name,
                    "page": page_no,
                    "text": part
                })

    if not chunks:
        return (
            "❌ PDF uploaded, but no readable text was found.",
            "",
            "",
            ""
        )

    # Create embeddings
    embeddings = model.encode(
        [x["text"] for x in chunks],
        normalize_embeddings=True
    )

    summary = "### 📄 Documents\n\n"

    for name, text in documents.items():

        sentences = re.split(
            r"(?<=[.!?])\s+",
            text.strip()
        )

        summary += (
            f"**{name}**\n"
            + " ".join(sentences[:4])
            + "\n\n"
        )

    key_points = "### 🔑 Key Points\n\n"

    for item in chunks[:8]:
        key_points += f"• {item['text'][:180]}...\n"

    questions = """### 💡 Suggested Questions\n\n- What is the main topic of this document?\n- What are the important points?\n- What problem is discussed?\n- What methods are used?\n- What are the advantages?\n- What are the limitations?\n- What is the conclusion?\n"""

    status = (
        f"✅ **PDF processing completed!**\n\n"
        f"📚 Documents: **{len(documents)}**\n"
        f"🧩 Chunks: **{len(chunks)}**"
    )

    return status, summary, key_points, questions


# =========================
# RETRIEVAL
# =========================

def retrieve(question, top_k=6):

    if embeddings is None:
        return []

    query_embedding = model.encode(
        [question],
        normalize_embeddings=True
    )[0]

    semantic = embeddings @ query_embedding

    q_words = get_words(question)

    keyword = np.array([
        len(q_words & get_words(x["text"]))
        / max(len(q_words), 1)
        for x in chunks
    ])

    scores = (
        0.75 * semantic +
        0.25 * keyword
    )

    indexes = np.argsort(scores)[::-1]

    results = []
    seen = set()

    for i in indexes:

        text = chunks[i]["text"]
        key = text[:100]

        if key in seen:
            continue

        seen.add(key)

        results.append({
            **chunks[i],
            "score": float(scores[i])
        })

        if len(results) >= top_k:
            break

    return results


# =========================
# ASK QUESTION
# =========================

def ask_question(question, history, mode):

    if not question.strip():
        return history, "", "Enter a question.", ""

    if not chunks:
        return (
            history,
            "",
            "❌ Upload and process a PDF first.",
            ""
        )

    results = retrieve(question)

    if not results:
        return (
            history,
            "",
            "No supporting information found.",
            ""
        )

    # Confidence
    confidence = max(
        x["score"] for x in results
    )

    if confidence >= 0.70:
        level = "🟢 HIGH"
    elif confidence >= 0.45:
        level = "🟡 MEDIUM"
    else:
        level = "🔴 LOW — possible hallucination risk"

    # Relevant sentences
    candidates = []

    for item in results:

        sentences = re.split(
            r"(?<=[.!?])\s+",
            item["text"]
        )

        for sentence in sentences:

            if len(sentence.split()) < 5:
                continue

            overlap = len(
                get_words(question)
                & get_words(sentence)
            )

            candidates.append(
                (overlap, sentence, item)
            )

    candidates.sort(
        key=lambda x: x[0],
        reverse=True
    )

    selected = candidates[:6]

    if not selected:
        selected = [
            (
                0,
                results[0]["text"],
                results[0]
            )
        ]

    # Answer modes
    if mode == "Short Answer":

        answer = " ".join(
            x[1] for x in selected[:2]
        )

    elif mode == "Bullet Points":

        answer = "\n".join(
            "• " + x[1]
            for x in selected
        )

    elif mode == "Exam / 10-Mark Answer":

        answer = (
            "### Answer\n\n"
            + "\n\n".join(
                x[1] for x in selected
            )
        )

    else:

        answer = (
            "### Answer\n\n"
            + " ".join(
                x[1] for x in selected
            )
        )

    answer += (
        f"\n\n### 🛡️ Hallucination Check\n"
        f"Confidence: **{level}** "
        f"({confidence:.2f})"
    )

    # Sources
    sources = "### 📚 Sources\n\n"

    for i, item in enumerate(results, 1):

        sources += (
            f"**[{i}] {item['file']} — "
            f"Page {item['page']}**\n"
            f"Similarity: `{item['score']:.2f}`\n\n"
        )

    # Analytics
    analytics = (
        "### 📊 Retrieval Analytics\n\n"
        f"- Retrieved chunks: **{len(results)}**\n"
        f"- Confidence: **{confidence:.2f}**\n"
        f"- Documents: **{len(documents)}**"
    )

    # Memory
    memory.append({
        "question": question,
        "answer": answer
    })

    if history is None:
        history = []

    history = history + [
        {
            "role": "user",
            "content": question
        },
        {
            "role": "assistant",
            "content": answer
        }
    ]

    return history, "", sources, analytics


# =========================
# PDF COMPARISON
# =========================

def compare_pdfs():

    names = list(documents.keys())

    if len(names) < 2:
        return "⚠️ Upload at least **2 PDFs**."

    result = "### 🔎 PDF Comparison\n\n"

    for i in range(len(names)):

        for j in range(i + 1, len(names)):

            a = get_words(documents[names[i]])
            b = get_words(documents[names[j]])

            result += (
                f"**{names[i]} vs {names[j]}**\n\n"
                f"- Common terms: **{len(a & b)}**\n"
                f"- Unique to first: **{len(a-b)}**\n"
                f"- Unique to second: **{len(b-a)}**\n\n"
            )

    return result


# =========================
# EXPORT CHAT
# =========================

def export_chat():

    if not memory:
        return None

    path = "/content/RAG_chat_export.txt"

    with open(
        path,
        "w",
        encoding="utf-8"
    ) as f:

        f.write(
            "HALLUCINATION-AWARE RAG ASSISTANT\n"
        )
        f.write("=" * 50 + "\n\n")

        for i, item in enumerate(memory, 1):

            f.write(
                f"QUESTION {i}\n"
                f"{item['question']}\n\n"
            )

            f.write(
                "ANSWER\n"
                f"{item['answer']}\n\n"
            )

            f.write("-" * 50 + "\n\n")

    return path


# =========================
# CLEAR CHAT
# =========================

def clear_chat():

    global memory
    memory = []

    return []


# =========================
# GRADIO APP
# =========================

with gr.Blocks(
    title="Hallucination-Aware RAG Assistant"
) as app:

    gr.Markdown("""
# 🧠 Hallucination-Aware RAG Assistant

### Upload your PDF → Ask questions → Get grounded answers with citations

**Features:** Multi-PDF • RAG • Hybrid Retrieval • Chat Memory •
Hallucination Check • Page Citations • PDF Comparison • Analytics • Export
""")

    # PDF UPLOAD
    pdf_upload = gr.File(
        label="📚 Upload PDF(s)",
        file_types=[".pdf"],
        file_count="multiple",
        type="filepath"
    )

    process_button = gr.Button(
        "🚀 Upload & Process PDFs",
        variant="primary"
    )

    status = gr.Markdown()

    with gr.Row():

        summary = gr.Markdown()

        key_points = gr.Markdown()

    suggested = gr.Markdown()

    gr.Markdown("## 💬 Ask Questions")

    answer_mode = gr.Dropdown(
        [
            "Short Answer",
            "Detailed Answer",
            "Bullet Points",
            "Exam / 10-Mark Answer"
        ],
        value="Detailed Answer",
        label="🎯 Answer Mode"
    )

    chatbot = gr.Chatbot(
        label="RAG Chat",
        # type="messages", # Removed the problematic 'type' argument
        height=400
    )

    question = gr.Textbox(
        label="❓ Ask about your PDF",
        placeholder="Example: What is the main topic of this document?"
    )

    with gr.Row():

        ask_button = gr.Button(
            "🔎 Ask",
            variant="primary"
        )

        clear_button = gr.Button(
            "🗑️ Clear Chat"
        )

        export_button = gr.Button(
            "📥 Export Chat"
        )

    sources = gr.Markdown()

    analytics = gr.Markdown()

    download = gr.File(
        label="📥 Download Chat"
    )

    gr.Markdown("## 🔎 Compare PDFs")

    compare_button = gr.Button(
        "Compare Uploaded PDFs"
    )

    comparison = gr.Markdown()

    # EVENTS

    process_button.click(
        upload_pdfs,
        inputs=pdf_upload,
        outputs=[
            status,
            summary,
            key_points,
            suggested
        ]
    )

    ask_button.click(
        ask_question,
        inputs=[
            question,
            chatbot,
            answer_mode
        ],
        outputs=[
            chatbot,
            question,
            sources,
            analytics
        ]
    )

    question.submit(
        ask_question,
        inputs=[
            question,
            chatbot,
            answer_mode
        ],
        outputs=[
            chatbot,
            question,
            sources,
            analytics
        ]
    )

    clear_button.click(
        clear_chat,
        outputs=chatbot
    )

    export_button.click(
        export_chat,
        outputs=download
    )

    compare_button.click(
        compare_pdfs,
        outputs=comparison
    )


app.launch(
    share=True,
    debug=True
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://110bcfe11478e1499d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
